# Загрузка датасета

In [59]:
import numpy as np
import pandas as pd

In [60]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
data = pd.read_excel("/content/drive/MyDrive/NetflixShows.xlsx")
data.head()

,title,rating,ratingLevel,ratingDescription,release year,user rating score,user rating size
0,White Chicks,PG-13,"crude and sexual humor, language and some drug content",80,2004,82.0,80
1,Lucky Number Slevin,R,"strong violence, sexual content and adult language",100,2006,NaN,82
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2016,98.0,80
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2008,98.0,80
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitable for all children.,70,2014,94.0,80


In [62]:
data.shape

(1000, 7)

In [63]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              1000 non-null   object 
 1   rating             1000 non-null   object 
 2   ratingLevel        941 non-null    object 
 3   ratingDescription  1000 non-null   int64  
 4   release year       1000 non-null   int64  
 5   user rating score  605 non-null    float64
 6   user rating size   1000 non-null   int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 54.8+ KB


In [64]:
data.isna().sum()

,0
title,0
rating,0
ratingLevel,59
ratingDescription,0
release year,0
user rating score,395
user rating size,0


# Разберемся с дубликатами

In [65]:
data.duplicated().sum()

np.int64(500)

In [66]:
data[data.duplicated()]

,title,rating,ratingLevel,ratingDescription,release year,user rating score,user rating size
50,Lucky Number Slevin,R,"strong violence, sexual content and adult language",100,2006,NaN,82
52,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2016,98.0,80
53,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2008,98.0,80
56,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitable for all children.,70,2014,94.0,80
57,Supernatural,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2016,95.0,80
...,...,...,...,...,...,...,...
991,Dawn of the Croods,TV-Y7,Suitable for children ages 7 and older,41,2017,72.0,80
992,Alpha and Omega: Dino Digs,TV-G,Suitable for all ages.,35,2016,NaN,82
995,The BFG,PG,"for action/peril, some scary moments and brief rude humor",60,2016,97.0,80
996,The Secret Life of Pets,PG,for action and some rude humor,60,2016,NaN,81


In [67]:
data = data.drop_duplicates()
data.duplicated().sum()

np.int64(0)

In [68]:
data.shape

(500, 7)

In [69]:
data.isna().sum()

,0
title,0
rating,0
ratingLevel,33
ratingDescription,0
release year,0
user rating score,244
user rating size,0


**Почему возникли дубли?**
Скорее всего при формировании датасета записи о фильмах и сериалах были продублированны (например при объединениии или выгрузке данных)

**Много ли их?**
Да, обнаружено 500 дублирующих строк (это 50% от всего датасета)

In [70]:
data_raw = pd.read_excel("/content/drive/MyDrive/NetflixShows.xlsx")
duplicates_by_rating = data_raw[data_raw.duplicated()].groupby('rating').size().sort_values(ascending=False)
duplicates_by_rating

,0
rating,
TV-14,128
PG,94
G,85
TV-MA,66
TV-Y,32
TV-PG,26
TV-G,23
TV-Y7-FV,19
TV-Y7,15


In [71]:
rating_stats = pd.DataFrame({
    'total_count': data_raw.groupby('rating').size(),
    'duplicates_count': data_raw[data_raw.duplicated()].groupby('rating').size()
}).fillna(0)

rating_stats['duplicates_share'] = rating_stats['duplicates_count'] / rating_stats['total_count']
rating_stats.sort_values(by='duplicates_count', ascending=False)

,total_count,duplicates_count,duplicates_share
rating,,,
TV-14,234,128.0,0.547009
PG,170,94.0,0.552941
G,138,85.0,0.615942
TV-MA,148,66.0,0.445946
TV-Y,68,32.0,0.470588
TV-PG,59,26.0,0.440678
TV-G,52,23.0,0.442308
TV-Y7-FV,44,19.0,0.431818
TV-Y7,38,15.0,0.394737


Больше всего дублирующихся записей оказалось в группах TV-14, PG и G. Это объясняется тем, что эти категории чаще встречаются в датасете, однако доля дублей внутри групп различается.

# Анализ признаков


In [72]:
data.head()

,title,rating,ratingLevel,ratingDescription,release year,user rating score,user rating size
0,White Chicks,PG-13,"crude and sexual humor, language and some drug content",80,2004,82.0,80
1,Lucky Number Slevin,R,"strong violence, sexual content and adult language",100,2006,NaN,82
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2016,98.0,80
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2008,98.0,80
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitable for all children.,70,2014,94.0,80


Всего в датасете 8 признаков. Сейчас каждый из них представлен типом object. Можно изучить каждый признак и закодировать его в нужный тип.
- title - текст
- rating — категориальный признак
- ratingLevel и ratingDescription судя по всему перепутаны местами, потому что не значения не соответствуют названиям. После замены:
  - ratingLevel - числовой признак
  - ratingDescription - текст
- release year — целое число
- user rating score — число
- user rating size — сходу не очень понятно что это значит. [Официальный датасет на kaggle](https://www.kaggle.com/datasets/edmund24/netflix-movie-and-tv-show-information-dataset) говорит, что это "The size of the user rating dataset, indicating the number of users who have rated the movie or TV show". Но если посмотреть на занчения, которые он принимает, то можно увидеть, что их всего 3 - 80, 81, 82. Причем во всех строках, где user rating size = 81 или 82 отсутствует  user rating score, а во всех строках где rating size = 80 - user rating score присутствует. Получается этот признак можно просто заменить на has_user_score

In [73]:
data = data.rename(columns={
    'ratingLevel': 'ratingDescription',
    'ratingDescription': 'ratingLevel'
})

data["user rating size"].unique()



array([80, 82, 81])

In [74]:
data.loc[data['user rating size'].isin([81, 82]), 'user rating score'].isna().all()


np.True_

In [75]:
data['has_user_score'] = data['user rating score'].notna()
data_without_size = data.drop(columns='user rating size')

In [76]:
data_without_size['title'] = data_without_size['title'].astype('string')
data_without_size['rating'] = data_without_size['rating'].astype('category')
data_without_size['ratingDescription'] = data_without_size['ratingDescription'].astype('string')
data_without_size['ratingLevel'] = data_without_size['ratingLevel'].astype('category')
data_without_size['release year'] = data_without_size['release year'].astype('Int64')
data_without_size['user rating score'] = pd.to_numeric(data_without_size['user rating score'], errors='coerce')

data_without_size.head()

,title,rating,ratingDescription,ratingLevel,release year,user rating score,has_user_score
0,White Chicks,PG-13,"crude and sexual humor, language and some drug content",80,2004,82.0,True
1,Lucky Number Slevin,R,"strong violence, sexual content and adult language",100,2006,NaN,False
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2016,98.0,True
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2008,98.0,True
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitable for all children.,70,2014,94.0,True


Надо решить что делать с пропусками. Они есть в столбцах ratingLevel и user rating score.

Разберемся для начала с ratingLevel. По смыслу этот признак похож на rating. Посмотрим есть ли там соответствие 1 к 1.

In [77]:
pd.set_option('display.max_colwidth', None)

data.groupby('rating', observed=True)['ratingDescription'].apply(lambda x: sorted(x.dropna().astype(object).unique().tolist())).reset_index(name='all_ratingDescription_values')

,rating,all_ratingDescription_values
0,G,[General Audiences. Suitable for all ages.]
1,NR,[This movie has not been rated.]
2,PG,"[Parental guidance suggested. May not be suitable for children., action and rude humor, action and some rude humor, action and violence throughout, and mild language, action sequences and peril, action violence, action violence throughout, some scary images, and language, action violence, language and rude humor, adult content and mld violence, animal action and humor, animated action violence, some scary cartoon images and mild language, brief mild language and some rude behavior, fantasy action, fantasy action and mild language, fantasy action/peril and some language, for action and some rude humor, for action, peril and brief language, for action/peril, some scary moments and brief rude humor, for rude humor and action, for some sequences of scary action and peril, for thematic material and language, intense scenes dealing with strong content, language, some comic violence and mild sex-related humor, martial arts action and some mild rude humor, mild action and rude humor, mild action and some rude humor, mild language and action sequences, mild language and comic action, mild sci-fi action, mild suggestive humor, mild thematic elements, mild thematic material and language, moderate adult language and intense action, rude and suggestive humor, and some action, rude humor and some action, scary and intense creature action and images, and for some rude humor, scary images, suggestive material, some language and smoking, sci-fi action violence throughout, brief language and momentary smoking, sequences of martial arts action, slapstick action and mild language, some action violence involving gunplay, and mild rude humor, some mild action, some mild language and rough hockey action, some mild thematic elements, some mild, rude humor, some peril, some reckless behavior, some rude dialogue, some rude humor, some rude humor and sports action, some rude humor, language and some scary action, some rude language and pranks, some scary action, rude humor and language, some scary images, some scary images and action, and brief mild language, some scary images, action and rude humor, some scary images, perilous action sequences and mild rude humor, some scary moments, some thematic elements, rude humor and action, thematic elements, an accident scene and some suggestive material, thematic elements, scary images, action and peril, thematic elements, scary images, some language and suggestive humor, violence and menacing action, rude humor, suggestive content and thematic elements]"
3,PG-13,"[For some rude and suggestive material, and for language., Parents strongly cautioned. May be inappropriate for children under 13., adult content, adult language and mild violence, crude and sexual humor, language and some drug content, crude and suggestive humor, and for language, language and some crude sexual humor, language and some sexual humor, some sex-related material, some violence and one sexual scene, some violent content and mature thematic elements, thematic elements, brief violence and innuendo, thematic material involving threatening behavior, and for violence and sexuality]"
4,R,"[Restricted. May be inappropriate for children 17 and under., bloody war violence, language throughout and some sexual material, language and brief violence, language, drug content, sexuality/nudity, and some violence-all involving teens, language, some drug use, violence and partial nudity, pervasive drug content and language, some violence and sexuality, pervasive language, some sexual material, violence and drug use, pervasive sexual content, brief graphic nudity, language and some drug use, some sexual material, some sexual material, and language throughout, strong crude sexual content, pervasive language, and drug use, strong sexual content and language, strong violence and language throughout, strong violence, sexual content an

Можно заметить, что одному rating может соответствовать несколько разных rating description.

Посмотрим на те группы в которых имеются пропуски и на описания, которые в этой группе встречаются.

In [78]:

ratings_with_na = (data_without_size.groupby('rating', observed=True)['ratingDescription'].apply(lambda x: x.isna().sum()).loc[lambda x: x > 0].index)

data_without_size[data_without_size['rating'].isin(ratings_with_na)].groupby('rating', observed=True)['ratingDescription'].value_counts(normalize=True, dropna=False).mul(100).round(2).rename('percent_within_rating').reset_index()

,rating,ratingDescription,percent_within_rating
0,G,General Audiences. Suitable for all ages.,98.11
1,G,<NA>,1.89
2,NR,This movie has not been rated.,70.00
3,NR,<NA>,30.00
4,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,94.34
5,TV-14,<NA>,4.72
6,TV-14,"dialogue, language, sexual situations and violence",0.94
7,TV-MA,For mature audiences. May not be suitable for children 17 and under.,73.17
8,TV-MA,<NA>,26.83
9,TV-PG,Parental guidance suggested. May not be suitable for all children.,93.94


Заметим, что в большинстве групп всего одно уникальное значение. Процент пропусков в каждой группе не превышает 30. Можно безопасно заменить пропуски на моду по категории.

In [79]:
data_without_size['ratingDescription'] = (data_without_size.groupby('rating', observed=True)['ratingDescription'].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else pd.NA)))
data_without_size['ratingDescription'].isna().sum()

np.int64(0)

In [80]:
data_without_size.head()

,title,rating,ratingDescription,ratingLevel,release year,user rating score,has_user_score
0,White Chicks,PG-13,"crude and sexual humor, language and some drug content",80,2004,82.0,True
1,Lucky Number Slevin,R,"strong violence, sexual content and adult language",100,2006,NaN,False
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2016,98.0,True
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2008,98.0,True
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitable for all children.,70,2014,94.0,True


Отлично, пропусков в ratingDescription не осталось. Переходим к user rating score.

In [81]:
data_without_size['user rating score'].isna().sum()

np.int64(244)

Пропусков здесь почти половина. Поэтому предлагается не заменять пропуски, а пользоваться данными из has_user_score признака при анализе.

In [82]:
data = data_without_size

Также для удобства переименуем поля release year и user rating score в те же самые имено, но с _ вместо пробелов между словами

In [83]:
data = data.rename(columns={'release year': 'release_year', 'user rating score': 'user_rating_score'})

Далее сгруппируем шоу по rating, определив каждый рейтинг к одной из групп: kids, family, teen, adult и unrated.

In [84]:
rating_to_group = {
    'TV-Y': 'kids',
    'TV-Y7': 'kids',
    'TV-Y7-FV': 'kids',

    'G': 'family',
    'TV-G': 'family',
    'PG': 'family',
    'TV-PG': 'family',

    'PG-13': 'teen',
    'TV-14': 'teen',

    'R': 'adult',
    'TV-MA': 'adult',

    'NR': 'unrated',
    'UR': 'unrated'
}

In [85]:
data['rating_age_group'] = data['rating'].map(rating_to_group)

Посмотрим на распределение полученных возрастных групп в датасете

In [86]:
data['rating_age_group'].value_counts().reset_index()

,rating_age_group,count
0,family,191
1,teen,118
2,adult,96
3,kids,84
4,unrated,11


In [87]:
# https://plotly.com/python/pie-charts/
import plotly.express as px

fig = px.pie(data['rating_age_group'].value_counts().reset_index(), values='count', names='rating_age_group', title='Распределение контента по возрастным группам', width=700)
fig.show()

Посмотрим на распределение пользовательской оценки

In [88]:
fig = px.box(data[data['has_user_score']], y='user_rating_score', width=700, height=800, title='Распределение пользовательской оценки user_rating_score')
fig.show()

In [89]:
mean_rating_by_group = data[data['has_user_score']].groupby('rating_age_group')['user_rating_score'].mean().reset_index().sort_values('user_rating_score', ascending=False)
mean_rating_by_group[mean_rating_by_group['rating_age_group'] != 'unrated']

,rating_age_group,user_rating_score
0,adult,84.893617
3,teen,81.674419
1,family,81.101010
2,kids,74.590909


In [90]:
fig = px.bar(mean_rating_by_group[mean_rating_by_group['rating_age_group'] != 'unrated'], x='rating_age_group', y='user_rating_score', width=700, title='Сравнение средней пользовательской оценки по возрастным группам')
fig.show()

# Внешний датасеты от IMDB

**IMDb** is the world's most popular and authoritative source for movie, TV and celebrity content. Find ratings and reviews for the newest movie and TV shows. (https://www.imdb.com)

Был использован официальный источник - https://developer.imdb.com/non-commercial-datasets/ (Страница с ссылками на сами датасеты https://datasets.imdbws.com/). Здесь прописано о Non-Commercial use, что подходит под наш проект.

Были взяты два датасета:
- title.basics.tsv.gz: базовый датасет, с названием, датой релиза, данными о продолжительности и жанрах.
- title.ratings.tsv.gz: датасет, который расширяет прошлый - добавляет пользовательский рейтинг на сайте.

Ниже импорт датасетов и их объединение в общий датасет по внутреннему id imdb.

In [91]:
title_basics = pd.read_csv('https://datasets.imdbws.com/title.basics.tsv.gz', sep='\t', na_values='\\N')

title_ratings = pd.read_csv('https://datasets.imdbws.com/title.ratings.tsv.gz', sep='\t', na_values='\\N')

/tmp/ipykernel_351/3464491763.py:1: DtypeWarning:

Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.



In [92]:
imdb = title_basics.merge(title_ratings, on='tconst', how='left')

В полученном датасете более 12млн строк. Для начала отфильтруем датасет - оставим только то, что выпущено не позже 2017 года.

In [93]:
imdb = imdb[imdb['startYear'] <= 2017]

Далее посмотрим на наличие дубликатов по паре primaryTitle + startYear (нужны будут далее для merge)

In [94]:
imdb.duplicated(subset=['primaryTitle', 'startYear']).sum()

np.int64(2626868)

Удалим такие дубликаты. Будем считать, что раз в этом датасете идет сортировка по возрастанию tconst (imdb id), то drop_duplictes просто оставит самую первую строку, а остальное, возможно, не основное шоу.

Например, в ходе поиска причин увеличения числа строк при left join (df.merge) и далее установления наличия дубликатов - был рассмотрел случайный пример такого случая с фильмом Dolphin Tale, где первая такая строка - сам фильм, а остальное какие-то tvEpisodes без рейтинга или с намного меньшим числом голосовавших.

In [95]:
imdb[imdb['primaryTitle'] == 'Dolphin Tale']

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
4033984,tt1564349,movie,Dolphin Tale,Dolphin Tale,0,2011.0,NaN,113.0,"Drama,Family",6.8,26788.0
5249495,tt2069117,tvEpisode,Dolphin Tale,Dolphin Tale,0,2011.0,NaN,NaN,Adventure,NaN,NaN
5254282,tt2072312,tvEpisode,Dolphin Tale,Dolphin Tale,0,2011.0,NaN,NaN,Adventure,NaN,NaN
7492399,tt3132654,tvEpisode,Dolphin Tale,Dolphin Tale,0,2013.0,NaN,26.0,"Adventure,Comedy,Drama",8.2,92.0


In [96]:
imdb_without_duplicates = imdb.drop_duplicates(subset=['primaryTitle', 'startYear'])

Далее смерджим наш исходный датасет после удаления дубликатов с новым imdb. Для идентификации записи будем использовать раннее упомянутую пару primaryTitle + startYear (в исходном датасете это title и release_year).

Перед этим посмотрим нет ли в исходном датасете дубликаты по паре title + release_year

In [97]:
data[data.duplicated(subset=['title', 'release_year'])]

,title,rating,ratingDescription,ratingLevel,release_year,user_rating_score,has_user_score,rating_age_group
449,Bordertown,TV-MA,For mature audiences. May not be suitable for children 17 and under.,110,2016,NaN,False,adult


Был найден один дубликат

In [98]:
data[data['title'] == 'Bordertown']

,title,rating,ratingDescription,ratingLevel,release_year,user_rating_score,has_user_score,rating_age_group
167,Bordertown,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2016,86.0,True,teen
449,Bordertown,TV-MA,For mature audiences. May not be suitable for children 17 and under.,110,2016,NaN,False,adult


Оказывается это два разных сериала, но с одинаковым названием. Уже после объединения датасетов вручную введем корректные данные для этих сериалов

In [99]:
data_imdb_merged = data.merge(imdb_without_duplicates, left_on=['title', 'release_year'], right_on=['primaryTitle', 'startYear'], how='left')

In [100]:
data_imdb_merged

,title,rating,ratingDescription,ratingLevel,release_year,user_rating_score,has_user_score,rating_age_group,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,White Chicks,PG-13,"crude and sexual humor, language and some drug content",80,2004,82.0,True,teen,tt0381707,movie,White Chicks,White Chicks,0.0,2004.0,NaN,109.0,"Comedy,Crime",5.9,191261.0
1,Lucky Number Slevin,R,"strong violence, sexual content and adult language",100,2006,NaN,False,adult,tt0425210,movie,Lucky Number Slevin,Lucky Number Slevin,0.0,2006.0,NaN,110.0,"Crime,Drama,Thriller",7.7,338146.0
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2016,98.0,True,teen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2008,98.0,True,teen,tt2161664,short,Prison Break,Prison Break,0.0,2008.0,NaN,NaN,"Comedy,Short",8.0,203.0
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitable for all children.,70,2014,94.0,True,family,tt3615240,tvEpisode,How I Met Your Mother,How I Met Your Mother,0.0,2014.0,NaN,43,Talk-Show,8.6,70.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,Russell Madness,PG,some rude humor and sports action,60,2015,NaN,False,family,tt4257950,movie,Russell Madness,Russell Madness,0.0,2015.0,NaN,92.0,"Family,Sport",4.3,787.0
496,Wiener Dog Internationals,G,General Audiences. Suitable for all ages.,35,2015,NaN,False,family,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497,Pup Star,G,General Audiences. Suitable for all ages.,35,2016,NaN,False,family,tt37149811,tvEpisode,Pup Star,Pup Star,0.0,2016.0,NaN,NaN,"News,Short",NaN,NaN
498,Precious Puppies,TV-G,Suitable for all ages.,35,2003,NaN,False,family,tt6500946,movie,Precious Puppies,Precious Puppies,0.0,2003.0,NaN,53.0,Documentary,6.1,81.0


Посмотрим какой из Bordertown надо исправить

In [101]:
data_imdb_merged[data_imdb_merged['title'] == 'Bordertown']

,title,rating,ratingDescription,ratingLevel,release_year,user_rating_score,has_user_score,rating_age_group,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
128,Bordertown,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2016,86.0,True,teen,tt3323254,tvSeries,Bordertown,Bordertown,0.0,2016.0,2016.0,21.0,"Animation,Comedy",5.6,3449.0
257,Bordertown,TV-MA,For mature audiences. May not be suitable for children 17 and under.,110,2016,NaN,False,adult,tt3323254,tvSeries,Bordertown,Bordertown,0.0,2016.0,2016.0,21.0,"Animation,Comedy",5.6,3449.0


In [102]:
imdb[(imdb['primaryTitle'] == 'Bordertown') & (imdb['startYear'] == 2016)]

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
8029056,tt3323254,tvSeries,Bordertown,Bordertown,0,2016.0,2016.0,21.0,"Animation,Comedy",5.6,3449.0
10117054,tt4937942,tvSeries,Bordertown,Sorjonen,0,2016.0,2020.0,60.0,"Crime,Drama,Mystery",7.6,13198.0
10459104,tt5711316,short,Bordertown,Bordertown,0,2016.0,NaN,5.0,"Documentary,History,Short",NaN,NaN


In [103]:
data_imdb_merged.iloc[257, 8:] = imdb[imdb['tconst'] == 'tt4937942'].iloc[0]

Добавим новый признак - has_imdb_data. Он указывает на то - были ли найдены данные из imdb и позволяет проще фильтровать датасет по наличию новых данных из внешних датасетов.

In [104]:
data_imdb_merged['has_imdb_data'] = data_imdb_merged['tconst'].notna()

Далее добавим еще один датасет. Его пришлось создать самому при помощи OMDb API
(The Open Movie Database) https://www.omdbapi.com/ - API сервис, позволяющий получить расширенные данные по IMDb id (в нашем датасете колонка tconst).

Для доступа к API был получен беспланый API ключ на 1000 запросов, выдаваемый на email (Данного объема хватит с учетом <500 шоу, которые удалось сопоставить с датасетом imdb)

Для начала импортируем библиотеку для работы с api запросами, укажем бесплатный API ключ и выделим список imdb ids, которые мы смогли сопоставить с исходным датасетом

In [105]:
import requests

omdb_api_key = '58a0041b'

imdb_ids = data_imdb_merged[data_imdb_merged['has_imdb_data']]['tconst'].to_list()

Пройдемся по каждому id, отправим запрос и сохраним его в общий список, который далее преобразуем в dataframe. Если случится ошибка при отправке запроса, выведем id на котором она произошла и остановим цикл (если вдруг кончится лимит по ключу или прочим причинам, чтобы далее понять какие шоу не успели получить)

In [106]:
# rows = []

# for imdb_id in imdb_ids:
#   try:
#     show = requests.get('https://www.omdbapi.com/', params={"apikey": omdb_api_key, "i": imdb_id, "r": "json"}).json()

#     if show.get('Response') != 'True':
#       continue

#     show['tconst'] = imdb_id
#     show['has_omdb_data'] = True
#     rows.append(show)

#   except Exception as e:
#     print('Ошибка отправки запроса для imdb_id:', imdb_id)
#     break

# omdb_df = pd.DataFrame(rows)
# omdb_df.to_csv('omdb.csv', index=False)

Для дальнейшей работы будем использовать экспортированный в файл датасет, т.к. каждый раз создавать его заново через OMDb API не получится. Ссылка на датасет: https://drive.google.com/file/d/1xThlNnYXCH4dX8ncKXAnaR_jgVDDyuqh/view?usp=sharing

In [107]:
omdb = pd.read_csv('/content/drive/MyDrive/omdb.csv')

Далее оставим в датасете только те поля, которые либо новые и не совпадают с imdb датасетом, либо те что могут найти применение, к примеру такие поля как Poster или Plot мы никак не сможем эффективно использовать в рамках нашего анализа.

In [108]:
omdb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288 entries, 0 to 287
Data columns (total 31 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Title          288 non-null    object 
 1   Year           288 non-null    object 
 2   Rated          226 non-null    object 
 3   Released       274 non-null    object 
 4   Runtime        247 non-null    object 
 5   Genre          284 non-null    object 
 6   Director       208 non-null    object 
 7   Writer         237 non-null    object 
 8   Actors         273 non-null    object 
 9   Plot           257 non-null    object 
 10  Language       243 non-null    object 
 11  Country        243 non-null    object 
 12  Awards         168 non-null    object 
 13  Poster         271 non-null    object 
 14  Ratings        288 non-null    object 
 15  Metascore      89 non-null     float64
 16  imdbRating     229 non-null    float64
 17  imdbVotes      239 non-null    object 
 18  imdbID    

In [109]:
omdb = omdb[['Rated', 'Released', 'Actors', 'Country', 'Awards', 'Metascore', 'tconst', 'has_omdb_data', 'totalSeasons']]

Объединим наш прошлый датасет с новым omdb датасетом

In [110]:
data_imdb_omdb_merged = data_imdb_merged.merge(omdb, on='tconst', how='left')

None значения в новом поле has_omdb_data, которое было создана при формировании датасета omdb, заменим на False

In [111]:
data_imdb_omdb_merged['has_omdb_data'] = data_imdb_omdb_merged['has_omdb_data'].fillna(False)

/tmp/ipykernel_351/27950386.py:1: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



In [112]:
data_imdb_omdb_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   title              500 non-null    object  
 1   rating             500 non-null    category
 2   ratingDescription  500 non-null    string  
 3   ratingLevel        500 non-null    category
 4   release_year       500 non-null    Int64   
 5   user_rating_score  256 non-null    float64 
 6   has_user_score     500 non-null    bool    
 7   rating_age_group   500 non-null    object  
 8   tconst             303 non-null    object  
 9   titleType          303 non-null    object  
 10  primaryTitle       303 non-null    object  
 11  originalTitle      303 non-null    object  
 12  isAdult            303 non-null    float64 
 13  startYear          303 non-null    float64 
 14  endYear            52 non-null     float64 
 15  runtimeMinutes     251 non-null    object  
 16  genres  

In [113]:
data_imdb_omdb_merged['titleType'].value_counts()

,count
titleType,
movie,132
tvEpisode,64
tvSeries,50
video,32
short,10
tvMovie,7
tvMiniSeries,4
tvSpecial,2
videoGame,2


In [114]:
data_imdb_omdb_merged.to_excel('merged_df.xlsx', index=False)

In [115]:
data_imdb_omdb_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   title              500 non-null    object  
 1   rating             500 non-null    category
 2   ratingDescription  500 non-null    string  
 3   ratingLevel        500 non-null    category
 4   release_year       500 non-null    Int64   
 5   user_rating_score  256 non-null    float64 
 6   has_user_score     500 non-null    bool    
 7   rating_age_group   500 non-null    object  
 8   tconst             303 non-null    object  
 9   titleType          303 non-null    object  
 10  primaryTitle       303 non-null    object  
 11  originalTitle      303 non-null    object  
 12  isAdult            303 non-null    float64 
 13  startYear          303 non-null    float64 
 14  endYear            52 non-null     float64 
 15  runtimeMinutes     251 non-null    object  
 16  genres  

Удалим теперь те признаки, которые подтянулись из датасетов imdb и аналогично с очисткой omdb - либо повторяют другие, либо не нужны.

In [116]:
data_imdb_omdb_merged = data_imdb_omdb_merged.drop(columns=['primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'Rated', 'Metascore', 'totalSeasons'])

In [117]:
data_imdb_omdb_merged.to_excel('new_netflix_df.xlsx')

Преобразуем признак Released в формат datetime

In [118]:
data_imdb_omdb_merged['Released'] = pd.to_datetime(data_imdb_omdb_merged['Released'])

Закодируем дополнительно столбец rating_age_group через Label Encoding. Возможно, это поможет далее увидеть корреляционные зависимости.

In [119]:
data_imdb_omdb_merged['rating_age_group_le'] = data_imdb_omdb_merged['rating_age_group'].map({'unrated': 0, 'kids': 1, 'family': 2, 'teen': 3, 'adult': 4})

Теперь посмотрим какие значения есть в столбце Country

In [120]:
data_imdb_omdb_merged['Country'].str.split(', ').explode().value_counts()

,count
Country,
United States,171
Canada,35
USA,31
United Kingdom,28
France,19
Japan,12
Australia,8
China,8
India,8


Заменим все USA на United States, UK на United Kingdom; выделим основную строку по средством взятия первого элемента из Country; сделаем параметр is_one_country, которые отмечает, что страна производства была одна.

In [121]:
data_imdb_omdb_merged['Country'] = data_imdb_omdb_merged['Country'].str.replace('USA', 'United States')
data_imdb_omdb_merged['Country'] = data_imdb_omdb_merged['Country'].str.replace('UK', 'United Kingdom')

data_imdb_omdb_merged['MainCountry'] = data_imdb_omdb_merged['Country'].str.split(', ').str[0]

data_imdb_omdb_merged['is_one_country'] = data_imdb_omdb_merged['Country'].str.split(', ').str.len() == 1

In [122]:
data_imdb_omdb_merged['Country'].str.split(', ').explode().value_counts()

,count
Country,
United States,202
Canada,35
United Kingdom,29
France,19
Japan,12
China,8
Australia,8
India,8
Germany,6


In [123]:
data_imdb_omdb_merged

,title,rating,ratingDescription,ratingLevel,release_year,user_rating_score,has_user_score,rating_age_group,tconst,titleType,...,numVotes,has_imdb_data,Released,Actors,Country,Awards,has_omdb_data,rating_age_group_le,MainCountry,is_one_country
0,White Chicks,PG-13,"crude and sexual humor, language and some drug content",80,2004,82.0,True,teen,tt0381707,movie,...,191261.0,True,2004-06-23,"Marlon Wayans, Shawn Wayans, Busy Philipps",United States,3 wins & 13 nominations total,True,3,United States,True
1,Lucky Number Slevin,R,"strong violence, sexual content and adult language",100,2006,NaN,False,adult,tt0425210,movie,...,338146.0,True,2006-04-07,"Josh Hartnett, Ben Kingsley, Morgan Freeman","United Kingdom, Germany, Canada, United States",5 wins & 4 nominations total,True,4,United Kingdom,False
2,Grey's Anatomy,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2016,98.0,True,teen,NaN,NaN,...,NaN,False,NaT,NaN,NaN,NaN,False,3,NaN,False
3,Prison Break,TV-14,Parents strongly cautioned. May be unsuitable for children ages 14 and under.,90,2008,98.0,True,teen,tt2161664,short,...,203.0,True,2008-01-10,"Chad Villella, Matt Bettinelli-Olpin, Rob Polonsky",United States,NaN,True,3,United States,True
4,How I Met Your Mother,TV-PG,Parental guidance suggested. May not be suitable for all children.,70,2014,94.0,True,family,tt3615240,tvEpisode,...,70.0,True,2014-03-27,"James Lipton, Josh Radnor, Cobie Smulders",NaN,NaN,True,2,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,Russell Madness,PG,some rude humor and sports action,60,2015,NaN,False,family,tt4257950,movie,...,787.0,True,2015-05-10,"Sean Giambrone, David Milchard, Steve Richmond",United States,NaN,True,2,United States,True
496,Wiener Dog Internationals,G,General Audiences. Suitable for all ages.,35,2015,NaN,False,family,NaN,NaN,...,NaN,False,NaT,NaN,NaN,NaN,False,2,NaN,False
497,Pup Star,G,General Audiences. Suitable for all ages.,35,2016,NaN,False,family,tt37149811,tvEpisode,...,NaN,True,NaT,NaN,NaN,NaN,False,2,NaN,False
498,Precious Puppies,TV-G,Suitable for all ages.,35,2003,NaN,False,family,tt6500946,movie,...,81.0,True,NaT,Lizzy Lovette,Australia,NaN,True,2,Australia,True


Добавим признаки has_win_award - была ли хотя бы одна победа и has_nomination_award - была ли хоты бя одна номинация (из столбца Awards)

In [124]:
data_imdb_omdb_merged['has_win_award'] = data_imdb_omdb_merged['Awards'].str.lower().str.contains('win')

In [125]:
data_imdb_omdb_merged['has_nomination_award'] = data_imdb_omdb_merged['Awards'].str.lower().str.contains('nominat')

In [126]:
data_imdb_omdb_merged.to_excel('new_netflix_df.xlsx')

Выделим основной жанр и общее количество представленных жанров для шоу из столбца genres

In [127]:
data_imdb_omdb_merged['genres'].str.split(',').explode().value_counts()

,count
genres,
Comedy,156
Animation,100
Adventure,94
Drama,84
Family,64
Action,36
Romance,23
Crime,20
Short,17


In [128]:
data_imdb_omdb_merged['primary_genre'] = data_imdb_omdb_merged['genres'].str.split(',').str[0]
data_imdb_omdb_merged['total_genres'] = data_imdb_omdb_merged['genres'].str.split(',').str.len()

In [129]:
data_imdb_omdb_merged['primary_genre'].value_counts()

,count
primary_genre,
Comedy,75
Adventure,69
Action,36
Animation,29
Drama,26
Documentary,13
Talk-Show,13
Crime,11
Biography,5


In [130]:
data_imdb_omdb_merged['total_genres'].value_counts()

,count
total_genres,
3.0,177
1.0,70
2.0,54


Под конец отнормируем наш user_rating_score (исходный Netflix рейтинг из выданного датасета) в 10 бальную шкалу, по аналогии с averageRating (рейтинг IMDb)

In [131]:
data_imdb_omdb_merged.loc[data_imdb_omdb_merged['has_user_score'], 'user_rating_score'] = data_imdb_omdb_merged.loc[data_imdb_omdb_merged['has_user_score'], 'user_rating_score'] / 10

Подготовленный датасет сохраним в удобное для работы название датафрейма - df

In [132]:
df = data_imdb_omdb_merged.copy()
df.to_excel('final_df.xlsx')

### Анализ трендов по возрастным категориям

Посмотрим на общее распределение возрастных категорий

In [133]:
# https://plotly.com/python/pie-charts/
import plotly.express as px

colors_map_rating_age_group = {
    'kids': '#E491C9',
    'family': '#982598',
    'teen': '#4b06a5',
    'adult': '#15173D',
    'unrated': '#BABABA'
}

fig = px.pie(df['rating_age_group'].value_counts().reset_index(), values='count', names='rating_age_group', color='rating_age_group', color_discrete_map=colors_map_rating_age_group, title='Распределение контента по возрастным группам', width=700)

fig.update_layout(legend_title_text='Возрастные категории') # https://plotly.com/python/legend/

fig.show()

Посмотрим по годам какое количество шоу снималось в каждой из категорий

In [134]:
df_grouped_age_year = df[df['rating_age_group'].notna()].groupby(['rating_age_group', 'release_year'])['title'].nunique().reset_index()

Ограничим выборку 21 веком, т.к. раннее ничего примечательного и отличного от периода 00х нету, а при таком ограничении проще смотреть на периоды активного роста.

In [135]:
fig = px.line(df_grouped_age_year[df_grouped_age_year['release_year'] >= 2000], x='release_year', y='title', color='rating_age_group', color_discrete_map=colors_map_rating_age_group, labels={'title': 'Количество релизов', 'rating_age_group': 'Возрастная группа', 'release_year': 'Год'}, title='Динамика количества выпущенных шоу по возрастным категориям в период с 2000 года')

# https://plotly.com/python/text-and-annotations/
fig.add_annotation(x=2015, y=19, showarrow=True, arrowhead=1, ax=-150, arrowsize=1, arrowwidth=2, text='2015 - год смены тренда лидеров по категориям')
fig.add_annotation(x=2016, y=50, showarrow=True, arrowhead=1, text='2016 - год с максимальным ростом')

fig.show()

Исходя из графика можно заметить, что по нашей выборке, до 2015 года самыми выпускаемым был контент с категориями, относящимися к family, a категория adult была самой непопулярной. Далее, с 2016 года, заметен резкий рост популярности шоу с рейтингами adult и teen, а прежний лидер family прибавил относительно других - немного. Тем самым, с 2016 года больше всего контента это adult и teen.

Говоря о шоу для детей, то его рост был почти равномерным и без сильных скачков на протяжении представленного временного интервала.

Также важно заметит, что в 2017 данные для всех категории начали сильное падение, возможно, в нашей выборке представлен не весь 2017 год.

Попробуем через доступный для некоторых строк столбец Released из датасета imdb - посмотреть распределение даты выпуска по месяцам:

In [136]:
df_has_imdb_data = df[df['has_imdb_data']].copy()
df_has_imdb_data['release_month'] = df_has_imdb_data['Released'].dt.month
df_2017_releases_count = df_has_imdb_data[df_has_imdb_data['release_year'] >= 2015].groupby(['release_month', 'release_year'])['title'].nunique().reset_index()

fig = px.bar(df_2017_releases_count, x='release_month', y='title', color='release_year', labels={'title': 'Количество релизов', 'release_month': 'Месяц', 'release_year': 'Год'}, title='Распределение числа релизов по месяцам для 2015-2017')

fig.update_xaxes(dtick=1) # https://plotly.com/python/tick-formatting/

fig.show()

Видно, что самих релизов довольно мало, основная часть это первые полгода и несколько в конце года. Сравним, для примера с 2015 и 2016 - как минимум в каждом месяце что-то выпущено, и хоть и 2015 не так насыщенный - все равно релизов в разы больше чем в 2017. Так что, скорее всего, 2017 действительно заполнен очень частично, с учетом такой разницы.

Но важно еще раз отметить, что это лишь часть выборки, на которой мы можем смотреть детализацию по месяцам, но общая картина ясна.

#### Вывод:
С 2016 года тренды изменились, больше контента теперь снимается в более взрослых категориях, категории family и kids увеличиваются лишь немного, в связи с общем увеличением числа выпущенного контента.

--------------------------------------------------------------------------------

In [137]:
df_grouped_year_age_rating = df[(df['averageRating'].notna()) & (df['rating_age_group'] != 'unrated') & (df['release_year'] >= 1985)].groupby(['release_year', 'rating_age_group'])['averageRating'].mean().reset_index()

fig = px.line(df_grouped_year_age_rating, x='release_year', y='averageRating', color='rating_age_group', title='Средний IMDb рейтинг по годам и возрастным категориям')
fig.show()

### Распределения рейтингов
Посмотрим как распределены по категориям рейтинги из исходного датасета - user_rating_score и из обогащенного - averageRating (рейтинг IMDb). Для большей честности возьмем подвыборку тех строк, на которые удалось добавить данные из IMDb.

In [138]:
df_has_imdb_data = df[df['has_imdb_data']].copy()

In [139]:
fig = px.box(df_has_imdb_data[(df_has_imdb_data['has_user_score']) & (df_has_imdb_data['rating_age_group'] != 'unrated')].sort_values('rating_age_group'), x='rating_age_group', y='user_rating_score', color='rating_age_group', labels={'rating_age_group': 'Возрастная категория', 'user_rating_score': 'Рейтинг'}, title='Распределение Netflix рейтинга по возрастным категориям')
fig.show()

In [140]:
fig = px.box(df_has_imdb_data[(df_has_imdb_data['averageRating'].notna()) & (df_has_imdb_data['rating_age_group'] != 'unrated')].sort_values('rating_age_group'), x='rating_age_group', y='averageRating', color='rating_age_group', labels={'rating_age_group': 'Возрастная категория', 'averageRating': 'Рейтинг'}, title='Распределение IMDb рейтинга по возрастным категориям')
fig.show()

## Гипотеза: влияние продолжительности на пользовательскую оценку

Проверим гипотезу: у фильмов и сериалов есть ли такой формат, при котором оценки зрителей выше.

Для фильмов будем смотреть продолжительность в минутах. Для сериалов возьмем количество сезонов. Так сравнение получается более честным: для фильмов важна длина фильма, а для сериалов — насколько длинным получился проект целиком.


In [141]:
data = data_imdb_omdb_merged.merge(
    omdb[['tconst', 'totalSeasons']].drop_duplicates(), on='tconst', how='left')

data['runtimeMinutes'] = pd.to_numeric(data['runtimeMinutes'], errors='coerce')
data['totalSeasons'] = pd.to_numeric(data['totalSeasons'], errors='coerce')

movies = data[data['titleType'].isin(['movie', 'tvMovie'])].copy()
movies = movies[movies['has_imdb_data']]
movies = movies[movies['has_user_score']]
movies = movies[movies['runtimeMinutes'].notna()]

series = data[data['titleType'].isin(['tvSeries', 'tvMiniSeries'])].copy()
series = series[series['has_imdb_data']]
series = series[series['has_user_score']]
series = series[series['totalSeasons'].notna()]

movies['group_bucket'] = pd.cut(movies['runtimeMinutes'], bins=[0, 80, 100, 120, np.inf],
                                labels=['<=80 min', '81-100 min', '101-120 min', '120+ min'])

series['group_bucket'] = pd.cut(series['totalSeasons'], bins=[0, 1, 3, np.inf],
                                labels=['1 season', '2-3 seasons', '4+ seasons'])

print('movies:', len(movies))
print('series:', len(series))


movies: 73
series: 15


Сначала посмотрим на короткие сводные таблицы. Они нужны, чтобы увидеть размер каждой группы и ее медианную оценку.


In [142]:
movie_summary = movies.groupby('group_bucket').agg(titles=('title', 'count'), median_user_rating=('user_rating_score', 'median')).reset_index()

series_summary = series.groupby('group_bucket').agg(titles=('title', 'count'), median_user_rating=('user_rating_score', 'median')).reset_index()

display(movie_summary)
display(series_summary)

/tmp/ipykernel_351/1996895075.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_351/1996895075.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,group_bucket,titles,median_user_rating
0,<=80 min,9,7.90
1,81-100 min,41,8.30
2,101-120 min,17,7.10
3,120+ min,6,7.75


,group_bucket,titles,median_user_rating
0,1 season,5,8.6
1,2-3 seasons,5,6.8
2,4+ seasons,5,6.8


На первом графике видно, как внутри каждой группы распределяются оценки. Для фильмов по оси X будет длина, а для сериалов — число сезонов.


In [143]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=2, subplot_titles=('Фильмы: длина в минутах', 'Сериалы: число сезонов'))

for bucket in ['<=80 min', '81-100 min', '101-120 min', '120+ min']:
    values = movies.loc[movies['group_bucket'] == bucket, 'user_rating_score']
    fig.add_trace(go.Box(y=values, name=bucket, marker_color='blue', showlegend=False), row=1, col=1)

for bucket in ['1 season', '2-3 seasons', '4+ seasons']:
    values = series.loc[series['group_bucket'] == bucket, 'user_rating_score']
    fig.add_trace(go.Box(y=values, name=bucket, marker_color='orange', showlegend=False), row=1, col=2)

fig.update_layout(title='Формат проекта и пользовательская оценка', width=1200, height=550)
fig.update_yaxes(title='user_rating_score')
fig.show()

На втором графике сравним медианные оценки. Это самый прямой способ понять, в какой группе оценки выше.


In [144]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Фильмы в минутах', 'Сериалы в сезонах'))

fig.add_trace(go.Bar(x=movie_summary['group_bucket'], y=movie_summary['median_user_rating'],
            text=movie_summary['titles'],marker_color='blue', showlegend=False),
            row=1, col=1)

fig.add_trace(
    go.Bar(x=series_summary['group_bucket'], y=series_summary['median_user_rating'],
           text=series_summary['titles'], marker_color='orange', showlegend=False),
            row=1, col=2)

fig.update_layout(title='Медианная пользовательская оценка по группам', width=1200, height=500)
fig.update_yaxes(title='Медианная пользовательская оценка')
fig.show()


Теперь посмотрим на IMDb-оценку рейтинга в таком же формате. Потом, мы сможем сравнить внутреннюю оценку Netflix и внешнюю оценку IMDb.


In [145]:
movie_imdb_summary = movies.groupby('group_bucket').agg(titles=('title', 'count'), median_imdb_rating=('averageRating', 'median')).reset_index()

series_imdb_summary = series.groupby('group_bucket').agg(titles=('title', 'count'), median_imdb_rating=('averageRating', 'median')).reset_index()

display(movie_imdb_summary)
display(series_imdb_summary)


/tmp/ipykernel_351/4079005521.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_351/4079005521.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,group_bucket,titles,median_imdb_rating
0,<=80 min,9,7.10
1,81-100 min,41,6.30
2,101-120 min,17,6.30
3,120+ min,6,7.05


,group_bucket,titles,median_imdb_rating
0,1 season,5,7.2
1,2-3 seasons,5,7.5
2,4+ seasons,5,8.0


Сначала посмотрим, как внутри групп распределяется рейтинг из IMDb.


In [146]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Фильмы: длина в минутах', 'Сериалы: число сезонов'))

for bucket in ['<=80 min', '81-100 min', '101-120 min', '120+ min']:
    values = movies.loc[movies['group_bucket']== bucket, 'averageRating']
    fig.add_trace(go.Box(y=values, name=bucket, marker_color='blue', showlegend=False), row=1, col=1)

for bucket in ['1 season', '2-3 seasons', '4+ seasons']:
    values = series.loc[series['group_bucket'] == bucket, 'averageRating']
    fig.add_trace(go.Box(y=values, name=bucket, marker_color='orange', showlegend=False), row=1, col=2)

fig.update_layout(title='Формат проекта и рейтинг IMDb', width=1200, height=550)
fig.update_yaxes(title='averageRating')
fig.show()

Теперь сравним медианные IMDb-оценки по группам. Так проще увидеть, где внешний рейтинг выше.


In [147]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Фильмы в минутах', 'Сериалы в сезонах'))

fig.add_trace(go.Bar(x=movie_imdb_summary['group_bucket'], y=movie_imdb_summary['median_imdb_rating'],
            text=movie_imdb_summary['titles'], marker_color='blue', showlegend=False),
            row=1, col=1)

fig.add_trace(go.Bar(x=series_imdb_summary['group_bucket'], y=series_imdb_summary['median_imdb_rating'],
            text=series_imdb_summary['titles'], marker_color='orange', showlegend=False),
            row=1, col=2)

fig.update_layout(title='Медианная IMDb-оценка по группам', width=1200, height=500)
fig.update_yaxes(title='Медианная IMDb-оценка')
fig.show()

### Выводы

- По рейтингу Нетфликса у фильмов лучше всего выглядит диапазон 81-100 минут, а у сериалов — проекты на 1 сезон
- По рейтингу из IMDb у фильмов чуть лучше выглядят более короткие и более длинные проекты, а у сериалов выше оценки у более длинных проектов с большим количеством сезонов
- Значит оценки Нетфликса и IMDb не дают один и тот же результат. Внутри какой-то платформы и во внешней среде один и тот же формат может не восприниматься одинаково
- Это важно: если смотреть только на одну метрику, можно сделать слишком простой вывод, какой формат лучше
- `Вывод для бизнеса`: если цель — сильнее понравиться зрителю внутри платформы, лучше выглядят фильмы средней длины и более короткие сериалы. Если цель, к примеру, внешний рейтинг и репутация на IMDb, картина может быть другой, это надо учитывать




## Гипотеза: влияние одной страны и международной копродукции на успех

Проверим гипотезу: есть ли разница между проектами одной страны и международными копродукциями. В качестве успеха будем смотреть две вещи: пользовательскую оценку и наличие наград.


Сначала подготовим данные. Оставим только проекты, где есть страна производства, пользовательская оценка и данные OMDb. После этого разделим  на две группы: одна страна и копродукция


In [148]:
country_data = data_imdb_omdb_merged.copy()
country_data = country_data[country_data['has_user_score']]
country_data = country_data[country_data['MainCountry'].notna()]
country_data = country_data[country_data['has_omdb_data']]

country_data['has_win_award'] = country_data['has_win_award'].fillna(0)
country_data['country_type'] = country_data['is_one_country'].map({True: 'one country', False: 'co-production'})

big_countries = country_data['MainCountry'].value_counts()
big_countries = big_countries[big_countries >= 5].index
country_top = country_data[country_data['MainCountry'].isin(big_countries)]


Сначала посмотрим на короткие таблицы. Первая для сравнения одной страны и копродукции, вторая показывает крупные страны.


In [149]:
country_data['has_win_award'] = pd.to_numeric(country_data['has_win_award'], errors='coerce').fillna(0)

big_countries = country_data['MainCountry'].value_counts()
country_top = country_data[country_data['MainCountry'].isin(big_countries[big_countries >= 5].index)]

country_type_summary = country_data.groupby('country_type').agg(titles=('title', 'size'), median_user_rating=('user_rating_score', 'median'),
                                            win_share=('has_win_award', 'mean')).reset_index()

country_rating_summary = country_top.groupby('MainCountry').agg(titles=('title', 'size'), median_user_rating=('user_rating_score', 'median'),
                                            win_share=('has_win_award', 'mean')).reset_index()

country_type_summary['win_share'] = (country_type_summary['win_share'] * 100).round(1)
country_rating_summary['win_share'] = (country_rating_summary['win_share'] * 100).round(1)

display(country_type_summary)
display(country_rating_summary)


,country_type,titles,median_user_rating,win_share
0,co-production,32,8.9,84.4
1,one country,76,7.9,46.1


,MainCountry,titles,median_user_rating,win_share
0,United Kingdom,6,8.2,66.7
1,United States,93,8.2,55.9


На первом графике сравним два типа проектов: одна страна и копродукция. Слева будет медианная оценка, справа — доля проектов с наградами.


In [150]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Медианная оценка', 'Доля проектов с наградами, %'))

fig.add_trace(go.Bar(x=country_type_summary['country_type'], y=country_type_summary['median_user_rating'],
                     text=country_type_summary['titles'], marker_color='blue', showlegend=False),row=1, col=1)

fig.add_trace(go.Bar( x=country_type_summary['country_type'], y=country_type_summary['win_share'],
                     text=country_type_summary['titles'], marker_color='orange',showlegend=False),row=1, col=2)

fig.update_layout(title='Одна страна и копродукция: два типа успеха', width=1200, height=450)
fig.update_yaxes(title='Медианная оценка')
fig.update_yaxes(title='Доля с наградами, %')
fig.show()

Теперь посмотрим на большие страны


In [151]:
fig = make_subplots(rows=1,cols=2, subplot_titles=('Медианная оценка', 'Доля проектов с наградами, %'))

fig.add_trace(go.Bar(x=country_rating_summary['MainCountry'], y=country_rating_summary['median_user_rating'],
                     text=country_rating_summary['titles'],marker_color='blue',showlegend=False),row=1, col=1)

fig.add_trace(go.Bar(x=country_rating_summary['MainCountry'], y=country_rating_summary['win_share'],
                     text=country_rating_summary['titles'], marker_color='orange', showlegend=False),row=1, col=2)

fig.update_layout(title='Крупные страны производства и два типа успеха', width=1200, height=450)
fig.update_yaxes(title='Медианная оценка', row=1, col=1)
fig.update_yaxes(title='Доля с наградами, %', row=1, col=2)
fig.show()

Теперь посмотрим на IMDb-оценку в таком же формате. Потом сравним внутреннюю оценку Netflix и внешний рейтинг IMDb


In [152]:
country_data['averageRating'] = pd.to_numeric(country_data['averageRating'], errors='coerce')

country_type_imdb_summary = country_data.groupby('country_type').agg(titles=('title', 'size'), median_imdb_rating=('averageRating', 'median')).reset_index()

country_rating_imdb_summary = country_top.groupby('MainCountry').agg(titles=('title', 'size'), median_imdb_rating=('averageRating', 'median')).reset_index()

display(country_type_imdb_summary)
display(country_rating_imdb_summary)


,country_type,titles,median_imdb_rating
0,co-production,32,6.95
1,one country,76,6.60


,MainCountry,titles,median_imdb_rating
0,United Kingdom,6,7.3
1,United States,93,6.6


Сначала сравним одну страну и копродукцию по IMDb-оценке. Справа оставим долю проектов с наградами, чтобы было удобно сопоставить рейтинг и репутационный успех.


In [153]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Медианная IMDb-оценка', 'Доля проектов с наградами, %'))

fig.add_trace(go.Bar(x=country_type_imdb_summary['country_type'], y=country_type_imdb_summary['median_imdb_rating'],
                     text=country_type_imdb_summary['titles'], marker_color='blue', showlegend=False), row=1, col=1)

fig.add_trace(go.Bar(x=country_type_summary['country_type'], y=country_type_summary['win_share'],
                     text=country_type_summary['titles'], marker_color='orange', showlegend=False), row=1, col=2)

fig.update_layout(title='Одна страна и копродукция: IMDb и награды', width=1200, height=450)
fig.update_yaxes(title='Медианная IMDb-оценка', row=1, col=1)
fig.update_yaxes(title='Доля с наградами, %', row=1, col=2)
fig.show()


Теперь посмотрим на крупные страны в таком же формате


In [154]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Медианная IMDb-оценка', 'Доля проектов с наградами, %'))

fig.add_trace(go.Bar(x=country_rating_imdb_summary['MainCountry'], y=country_rating_imdb_summary['median_imdb_rating'],
                     text=country_rating_imdb_summary['titles'], marker_color='blue', showlegend=False), row=1, col=1)

fig.add_trace(go.Bar(x=country_rating_summary['MainCountry'], y=country_rating_summary['win_share'],
                     text=country_rating_summary['titles'], marker_color='orange', showlegend=False), row=1, col=2)

fig.update_layout(title='Крупные страны производства: IMDb и награды', width=1200, height=450)
fig.update_yaxes(title='Медианная IMDb-оценка', row=1, col=1)
fig.update_yaxes(title='Доля с наградами, %', row=1, col=2)
fig.show()


### Выводы

- По рейтингам Нетфликса и IMDb вывод в целом совпадает: копродукции сильнее, чем проекты одной страны
- У копродукций выше медианная оценка и доля проектов с наградами (Нетфликс и IMDb)
- `Вывод для бизнеса`: международные копродукции могут быть интересны не только с точки зрения престижа и наград, но и с точки зрения зрительского отклика. То есть — больше шансов рассчитывать на репутационный, и пользовательский успех, при таком формате.


## Гипотеза: основной жанр связан с успехом шоу

Сначала соберем сводную таблицу, где для жанров, с количеством шоу от 10, посчитаем медианные рейтинги (для нашего малообъемного датасета медиана может быть понадежнее среднего) и долю шоу, с победами и номинациями на различные награды.

In [155]:
df_has_genres_and_rating = df[(df['primary_genre'].notna()) & (df['averageRating'].notna()) & (df['has_user_score'])]
genre_summary = df_has_genres_and_rating.groupby('primary_genre').agg(
    titles=('title', 'nunique'),
    median_user_rating=('user_rating_score', 'median'),
    median_imdb_rating=('averageRating', 'median'),
    win_share=('has_win_award', 'mean'),
    nomination_share=('has_nomination_award', 'mean')
).reset_index()

genre_summary['win_share'] = genre_summary['win_share'] * 100
genre_summary['nomination_share'] = genre_summary['nomination_share'] * 100

genre_summary = genre_summary[genre_summary['titles'] >= 10].sort_values('median_imdb_rating', ascending=False)
genre_summary

,primary_genre,titles,median_user_rating,median_imdb_rating,win_share,nomination_share
8,Drama,11,6.8,7.3,100.0,100.0
2,Adventure,29,8.2,6.9,76.923077,100.0
0,Action,15,9.1,6.7,91.666667,100.0
5,Comedy,31,7.9,6.2,66.666667,100.0


Во-первый сразу отметим, что доля номинированных везде 100%, а значит рассматривать этот критерий не имеет смысла

In [156]:
genre_summary = genre_summary.drop(columns=['nomination_share'])
genre_summary

,primary_genre,titles,median_user_rating,median_imdb_rating,win_share
8,Drama,11,6.8,7.3,100.0
2,Adventure,29,8.2,6.9,76.923077
0,Action,15,9.1,6.7,91.666667
5,Comedy,31,7.9,6.2,66.666667


In [157]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=3, subplot_titles=('Медианная IMDb-оценка', 'Медианная Netflx-оценка', 'Доля проектов с наградами, %'))

fig.add_trace(go.Bar(
    x=genre_summary['primary_genre'],
    y=genre_summary['median_imdb_rating'],
    text=genre_summary['titles'],
    marker_color='blue',
    showlegend=False
), row=1, col=1)

fig.add_trace(go.Bar(
    x=genre_summary['primary_genre'],
    y=genre_summary['median_user_rating'],
    text=genre_summary['titles'],
    marker_color='red',
    showlegend=False
), row=1, col=2)

fig.add_trace(go.Bar(
    x=genre_summary['primary_genre'],
    y=genre_summary['win_share'],
    text=genre_summary['titles'],
    marker_color='orange',
    showlegend=False
), row=1, col=3)

fig.update_layout(title='Основной жанр и два типа успеха', width=1500, height=500)
fig.update_yaxes(title='Медианная IMDb-оценка', row=1, col=1)
fig.update_yaxes(title='Медианная Netflix-оценка', row=1, col=2)
fig.update_yaxes(title='Доля проектов с наградами, %', row=1, col=3)

fig.show()

Отметим, что сравнивая исходные рейтинги и IMDb - между ними замечена практически отрицательная зависимость, где если смотреть по IMDb жанр Drama у нас наилучший, то по Netflix оценке ровно наоборот. Единственное, что Comedy в обоих случаях имеют примерно равную оценку.

Если смотреть на 3-й график о доле проектов с наградами, то тут градация схожа с оценкой IMDb, кроме выходящего из нисходящего тренда, как у медианной оценки IMDb, жанра Action.

Для наглядности посмотрим еще на диаграммы рассеяния по парам оценка (для каждой оценки свой график) + доля проектов с наградами.

In [158]:
fig = px.scatter(
    genre_summary,
    x='median_imdb_rating',
    y='win_share',
    size='titles',
    color='primary_genre',
    text='primary_genre',
    title='Распределение жанров по оценке IMDb и наградам, с демонастрацией масштаба присутствия в каталоге',
    labels={
        'median_imdb_rating': 'Медианная IMDb-оценка',
        'win_share': 'Доля проектов с наградами, %',
        'titles': 'Количество тайтлов',
        'primary_genre': 'Основной жанр'
    },
    width=1100,
    height=700
)

fig.update_traces(textposition='top center')
fig.show()

In [159]:
fig = px.scatter(
    genre_summary,
    x='median_user_rating',
    y='win_share',
    size='titles',
    color='primary_genre',
    text='primary_genre',
    title='Распределение жанров по оценке Netflix и наградам, с демонастрацией масштаба присутствия в каталоге',
    labels={
        'median_user_rating': 'Медианная Netflix-оценка',
        'win_share': 'Доля проектов с наградами, %',
        'titles': 'Количество тайтлов',
        'primary_genre': 'Основной жанр'
    },
    width=1100,
    height=700
)

fig.update_traces(textposition='top center')
fig.show()

Из этих первичных наблюдей можно предположить, что, судя по нашему датасету, на нетфликсе ценится в большей степени жанр Action, а Drama наоброт - вопреки оценкам на IMDb здесь оценивается хуже всех. Adventure везде стабильно неплох, а Comedy, несмотря на низкий рейтинг по IMDb, на Netflix, относительно других, не сильно хуже.